In [1]:
import pandas as pd
import sqlite3
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

PROCESSED_PATH = "../data/processed/"
DB_PATH = "../data/processed/ecommerce_olist.db"

In [2]:
main_df = pd.read_csv(PROCESSED_PATH + "ecommerce_main_dataset.csv")
kpi_summary = pd.read_csv(PROCESSED_PATH + "kpi_summary.csv")
monthly_revenue = pd.read_csv(PROCESSED_PATH + "monthly_revenue.csv")
top_categories = pd.read_csv(PROCESSED_PATH + "top_product_categories.csv")
top_states = pd.read_csv(PROCESSED_PATH + "top_customer_states.csv")
payment_distribution = pd.read_csv(PROCESSED_PATH + "payment_distribution.csv")
review_distribution = pd.read_csv(PROCESSED_PATH + "review_distribution.csv")
delivery_summary = pd.read_csv(PROCESSED_PATH + "delivery_summary.csv")
customer_rfm = pd.read_csv(PROCESSED_PATH + "customer_rfm_analysis.csv")
customer_segment_summary = pd.read_csv(PROCESSED_PATH + "customer_segment_summary.csv")

print("All processed files loaded successfully!")
print("Main dataset shape:", main_df.shape)

All processed files loaded successfully!
Main dataset shape: (113425, 42)


In [3]:
conn = sqlite3.connect(DB_PATH)

main_df.to_sql("ecommerce_main", conn, if_exists="replace", index=False)
kpi_summary.to_sql("kpi_summary", conn, if_exists="replace", index=False)
monthly_revenue.to_sql("monthly_revenue", conn, if_exists="replace", index=False)
top_categories.to_sql("top_product_categories", conn, if_exists="replace", index=False)
top_states.to_sql("top_customer_states", conn, if_exists="replace", index=False)
payment_distribution.to_sql("payment_distribution", conn, if_exists="replace", index=False)
review_distribution.to_sql("review_distribution", conn, if_exists="replace", index=False)
delivery_summary.to_sql("delivery_summary", conn, if_exists="replace", index=False)
customer_rfm.to_sql("customer_rfm_analysis", conn, if_exists="replace", index=False)
customer_segment_summary.to_sql("customer_segment_summary", conn, if_exists="replace", index=False)

conn.close()

print("SQLite database created successfully!")
print("Database saved to:", DB_PATH)

SQLite database created successfully!
Database saved to: ../data/processed/ecommerce_olist.db


In [4]:
conn = sqlite3.connect(DB_PATH)

In [5]:
query = """
SELECT
    ROUND(SUM(payment_value), 2) AS total_revenue,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_unique_id) AS total_customers
FROM ecommerce_main
WHERE order_status = 'delivered';
"""

pd.read_sql_query(query, conn)

,total_revenue,total_orders,total_customers
0,19776160.44,96478,93358


In [6]:
query = """
SELECT
    order_year_month,
    ROUND(SUM(payment_value), 2) AS total_revenue,
    COUNT(DISTINCT order_id) AS total_orders
FROM ecommerce_main
WHERE order_status = 'delivered'
GROUP BY order_year_month
ORDER BY order_year_month;
"""

pd.read_sql_query(query, conn)

,order_year_month,total_revenue,total_orders
0,2016-09,NaN,1
1,2016-10,61746.94,265
2,2016-12,19.62,1
3,2017-01,176491.49,750
4,2017-02,325782.66,1653
5,2017-03,505735.83,2546
6,2017-04,456108.32,2303
7,2017-05,701313.60,3546
8,2017-06,585400.98,3135
9,2017-07,716069.98,3872


In [7]:
query = """
SELECT
    product_category_name_english,
    ROUND(SUM(payment_value), 2) AS total_revenue,
    COUNT(DISTINCT order_id) AS total_orders
FROM ecommerce_main
WHERE order_status = 'delivered'
  AND product_category_name_english IS NOT NULL
GROUP BY product_category_name_english
ORDER BY total_revenue DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,product_category_name_english,total_revenue,total_orders
0,bed_bath_table,1692714.28,9272
1,health_beauty,1620684.04,8647
2,computers_accessories,1549372.59,6530
3,furniture_decor,1394466.93,6307
4,watches_gifts,1387362.45,5495
5,sports_leisure,1349446.93,7530
6,housewares,1069787.97,5743
7,auto,833745.67,3810
8,garden_tools,810614.93,3448
9,cool_stuff,744649.32,3559


In [8]:
query = """
SELECT
    customer_state,
    ROUND(SUM(payment_value), 2) AS total_revenue,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_unique_id) AS total_customers
FROM ecommerce_main
WHERE order_status = 'delivered'
GROUP BY customer_state
ORDER BY total_revenue DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,customer_state,total_revenue,total_orders,total_customers
0,SP,7403993.29,40501,39156
1,RJ,2688933.90,12350,11917
2,MG,2281229.16,11354,11001
3,RS,1110976.47,5345,5168
4,PR,1030822.39,4923,4769
5,BA,773182.02,3256,3158
6,SC,767093.97,3546,3449
7,GO,493068.70,1957,1895
8,DF,421374.86,2080,2019
9,ES,398321.90,1995,1928


In [9]:
query = """
SELECT
    CASE 
        WHEN is_late_delivery = 1 THEN 'Late'
        ELSE 'On Time / Early'
    END AS delivery_status,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(AVG(review_score), 2) AS avg_review_score,
    ROUND(AVG(delivery_time_days), 2) AS avg_delivery_time_days
FROM ecommerce_main
WHERE order_status = 'delivered'
GROUP BY is_late_delivery;
"""

pd.read_sql_query(query, conn)

,delivery_status,total_orders,avg_review_score,avg_delivery_time_days
0,On Time / Early,88652,4.21,10.39
1,Late,7826,2.55,30.89


In [10]:
query = """
SELECT
    customer_segment,
    total_customers,
    ROUND(avg_recency, 2) AS avg_recency,
    ROUND(avg_frequency, 2) AS avg_frequency,
    ROUND(avg_monetary, 2) AS avg_monetary,
    ROUND(avg_rfm_score, 2) AS avg_rfm_score
FROM customer_segment_summary
ORDER BY total_customers DESC;
"""

pd.read_sql_query(query, conn)

,customer_segment,total_customers,avg_recency,avg_frequency,avg_monetary,avg_rfm_score
0,Regular Customer,24636,185.83,1.01,172.42,9.18
1,At Risk Customer,22230,394.66,1.05,212.11,8.50
2,Lost Customer,14986,395.51,1.00,214.80,5.95
3,New Customer,14984,90.88,1.00,204.36,8.99
4,Loyal Customer,8546,97.21,1.02,81.78,10.93
5,High Value Customer,7976,93.14,1.20,480.61,13.55


In [11]:
query = """
SELECT
    customer_unique_id,
    recency,
    frequency,
    ROUND(monetary, 2) AS monetary,
    R_score,
    F_score,
    M_score,
    RFM_total_score,
    customer_segment
FROM customer_rfm_analysis
WHERE customer_segment = 'High Value Customer'
ORDER BY monetary DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)

,customer_unique_id,recency,frequency,monetary,R_score,F_score,M_score,RFM_total_score,customer_segment
0,ef8d54b3797ea4db1d63f0ced6a906e9,133,1,30186.00,4,5,5,14,High Value Customer
1,763c8b1c9c68a0229c42c9fc6f662b93,46,1,29099.52,5,3,5,13,High Value Customer
2,c8460e4251689ba205045f3ea17884a1,22,4,27935.46,5,5,5,15,High Value Customer
3,eae0a83d752b1dd32697e0e7b4221656,127,2,25051.89,4,5,5,14,High Value Customer
4,adfa1cab2b2c8706db21bb13c0a1beb1,89,1,19457.04,5,4,5,14,High Value Customer
5,fff5eb4918b2bf4b2da476788d42051c,58,1,17069.76,5,5,5,15,High Value Customer
6,be825ddd3b40db3f91bf05b4e9435d56,79,1,12490.88,5,4,5,14,High Value Customer
7,906a8a4ec9f3d4c3e64fa6d1c4fe6009,41,2,11881.01,5,5,5,15,High Value Customer
8,c8ed31310fc440a3f8031b177f9842c3,18,1,11572.80,5,4,5,14,High Value Customer
9,fe2b2f70f3dc31c23319ae1029eac77f,173,1,11531.41,4,5,5,14,High Value Customer


In [12]:
conn.close()
print("Database connection closed.")

Database connection closed.


In [13]:
import os

db_file = "../data/processed/ecommerce_olist.db"

if os.path.exists(db_file):
    print("Database berhasil dibuat!")
    print("Path:", db_file)
    print("Size:", os.path.getsize(db_file), "bytes")
else:
    print("Database belum ditemukan.")

Database berhasil dibuat!
Path: ../data/processed/ecommerce_olist.db
Size: 59265024 bytes


In [14]:
import sqlite3
import pandas as pd

DB_PATH = "../data/processed/ecommerce_olist.db"

conn = sqlite3.connect(DB_PATH)
print("Database connected successfully!")

Database connected successfully!


In [15]:
query = """
SELECT name 
FROM sqlite_master 
WHERE type='table';
"""

tables = pd.read_sql_query(query, conn)
tables

,name
0,ecommerce_main
1,kpi_summary
2,monthly_revenue
3,top_product_categories
4,top_customer_states
5,payment_distribution
6,review_distribution
7,delivery_summary
8,customer_rfm_analysis
9,customer_segment_summary


In [16]:
query = """
SELECT
    ROUND(SUM(payment_value), 2) AS total_revenue,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_unique_id) AS total_customers,
    ROUND(SUM(payment_value) / COUNT(DISTINCT order_id), 2) AS average_order_value
FROM ecommerce_main
WHERE order_status = 'delivered';
"""

pd.read_sql_query(query, conn)

,total_revenue,total_orders,total_customers,average_order_value
0,19776160.44,96478,93358,204.98


In [17]:
query = """
SELECT
    product_category_name_english,
    ROUND(SUM(payment_value), 2) AS total_revenue,
    COUNT(DISTINCT order_id) AS total_orders
FROM ecommerce_main
WHERE order_status = 'delivered'
  AND product_category_name_english IS NOT NULL
GROUP BY product_category_name_english
ORDER BY total_revenue DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,product_category_name_english,total_revenue,total_orders
0,bed_bath_table,1692714.28,9272
1,health_beauty,1620684.04,8647
2,computers_accessories,1549372.59,6530
3,furniture_decor,1394466.93,6307
4,watches_gifts,1387362.45,5495
5,sports_leisure,1349446.93,7530
6,housewares,1069787.97,5743
7,auto,833745.67,3810
8,garden_tools,810614.93,3448
9,cool_stuff,744649.32,3559


In [18]:
conn.close()
print("Database connection closed.")

Database connection closed.
